In [1]:
# 파이썬의 모든 자료형은 인스턴스
# iterable은 반복 돌릴 수 있는 인스턴스 
# 데이터 작업을 위한 기본 요소 - torch.utils.data, torch.utils.data.Dataset
# Dataset은 샘플과 정답(label)을 저장
# 여기서 샘플은 입력 데이터를 뜻하고 정답은 말 그대로 사람이 지정한 정답
# DataLoader는 Dataset을 순회 가능한 객체(iterable)로 감싼다.

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# 공개 데이터셋에서 학습 데이터를 내려받기
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# 공개 데이터셋에서 테스트 데이터를 내려받기
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
batch_size = 64

# 데이터로더 생성
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# test_dataloader에서 하나씩 꺼낸 값을 구조분해할당
for X, y in test_dataloader:
  print(f"Shape of X [N, C, H, W]: {X.shape}")
  print(f"Shape of y: {y.shape} {y.dtype}")
  break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
# 학습에 사용할 CPU나 GPU, MPS 장치를 얻기
# 삼항 중첩
device = (
	"cuda"
	if torch.cuda.is_available()
	else "mps"
	if torch.backends.mps.is_available()
	else "cpu"
)

print(f"Using {device} device")

# 모델 정의
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28*28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
  
model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# 모델학습을 위한 손실함수와 옵티마이저 생성
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [6]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    # 예측 오류 계산
    pred = model(X)
    loss = loss_fn(pred, y)

    # 역전파
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")


Epoch 1
-------------------------------
loss: 2.306709  [   64/60000]
loss: 2.297334  [ 6464/60000]
loss: 2.278054  [12864/60000]
loss: 2.267066  [19264/60000]
loss: 2.259938  [25664/60000]
loss: 2.227279  [32064/60000]
loss: 2.230553  [38464/60000]
loss: 2.198778  [44864/60000]
loss: 2.188343  [51264/60000]
loss: 2.166876  [57664/60000]
Test Error: 
 Accuracy: 50.7%, Avg loss: 2.158813 

Epoch 2
-------------------------------
loss: 2.169652  [   64/60000]
loss: 2.162373  [ 6464/60000]
loss: 2.102348  [12864/60000]
loss: 2.112510  [19264/60000]
loss: 2.078818  [25664/60000]
loss: 2.016498  [32064/60000]
loss: 2.041307  [38464/60000]
loss: 1.964030  [44864/60000]
loss: 1.955855  [51264/60000]
loss: 1.894714  [57664/60000]
Test Error: 
 Accuracy: 60.7%, Avg loss: 1.889141 

Epoch 3
-------------------------------
loss: 1.926323  [   64/60000]
loss: 1.896361  [ 6464/60000]
loss: 1.772072  [12864/60000]
loss: 1.804106  [19264/60000]
loss: 1.713175  [25664/60000]
loss: 1.662315  [32064/600